# lindblad_map — Lindblad (secular Redfield / GKSL) for the enhanced algorithm

Uses the jump-operator construction from the Master's thesis (S. Mader, Sec. 6; `calc_L_list` in
`figure6.21.py`). Each site couples to a Drude bath; the jump operators are built in the system
eigenbasis with rates set by the one-sided bath spectrum $2\,\mathrm{Re}\,\tau(\omega)$ (detailed
balance).

Lindblad needs no map file either: GKSL is a semigroup, so the enhanced algorithm only ever needs the
ONE-STEP map
$$\mathcal L(\Delta t)=e^{L_{\rm Lindblad}\Delta t}$$
— a single matrix exponential. Its Kraus operators $M_k(\Delta t)$, built once by `grid.enhanced_kraus`,
then reproduce the whole trajectory exactly, because $\mathcal L(n\Delta t)=\mathcal L(\Delta t)^n$
holds identically.

**Provides:**
- `calc_L_list(...)` → the GKSL jump operators $L_k$ (thesis method)
- `lindblad_superop(H, L_list)` → the Liouvillian $L$ (column-stacking)
- `lindblad_onestep_map(dt_fs)` → $\mathcal L(\Delta t)$ (the whole classical cost)
- `standard_lindblad_rho(...)` → populations from a direct qutip `mesolve` (the benchmark)

In [ ]:
import numpy as np
from scipy.linalg import expm

In [ ]:
def calc_L_list(H, lam, gamma, T_K, KB_CM):
    """GKSL jump operators, thesis construction (figure6.21.py:calc_L_list).
    One local site-projector coupling per site; rate 2 Re tau(omega)."""
    kBT = KB_CM * T_K
    d = H.shape[0]

    def Re_tau(omega):
        if abs(omega) < 1e-12:
            return 2 * kBT * lam / gamma                       # cm^-1
        J = 2 * lam * omega * gamma / (omega ** 2 + gamma ** 2)
        n = 1.0 / (np.exp(omega / kBT) - 1.0)
        return J * (n + 1.0)

    eks, V = np.linalg.eigh(H)                                 # V[:,M] = |v_M>
    # Bohr frequencies: all ordered pairs (M != N), plus 0 (pure dephasing)
    w_diff = [eks[M] - eks[N] for M in range(d) for N in range(d)
              if M != N] + [0.0]

    L_list = []
    for m in range(d):                                        # site projector
        for w in w_diff:
            Lmw = np.zeros((d, d), dtype=complex)
            for M in range(d):
                for N in range(d):
                    if abs((eks[M] - eks[N]) - w) < 1e-9:
                        amp = np.conj(V[m, M]) * V[m, N]       # <v_M|m><m|v_N>
                        Lmw += amp * np.outer(V[:, M], np.conj(V[:, N]))
            gam = 2 * Re_tau(w)
            if gam > 0:
                L_list.append(np.sqrt(gam) * Lmw)
    return L_list

In [ ]:
def lindblad_superop(H, L_list):
    """Liouvillian L (D x D, D=d^2) in the column-stacking convention:
    vec(A rho B) = (B^T (x) A) vec(rho)."""
    d = H.shape[0]
    I = np.eye(d)
    Lsup = -1j * (np.kron(I, H) - np.kron(H.T, I))            # -i[H, .]
    for Lk in L_list:
        LdL = Lk.conj().T @ Lk
        Lsup += (np.kron(Lk.conj(), Lk)
                 - 0.5 * np.kron(I, LdL)
                 - 0.5 * np.kron(LdL.T, I))
    return Lsup

In [ ]:
def lindblad_onestep_map(dt_fs, *, H, lam, gamma, T_K, KB_CM, FS_TO_CM):
    """The one-step map L(dt) = exp(L_Lindblad * dt) -- a single matrix
    exponential, the entire classical cost of the enhanced algorithm here."""
    Lsup = lindblad_superop(H, calc_L_list(H, lam, gamma, T_K, KB_CM))
    return expm(Lsup * (dt_fs * FS_TO_CM))

In [ ]:
def standard_lindblad_rho(t_fs, rho0, *, H, lam, gamma, T_K, KB_CM, FS_TO_CM):
    """BENCHMARK: dynamics from a direct qutip mesolve with the same L_list --
    the ordinary implementation, no grid, no Kraus operators. Returns the FULL
    density matrices (nt, d, d) so populations AND coherences can be compared."""
    import qutip as qt
    L_list = calc_L_list(H, lam, gamma, T_K, KB_CM)
    res = qt.mesolve(qt.Qobj(H), qt.Qobj(np.asarray(rho0, complex)),
                     np.asarray(t_fs) * FS_TO_CM,
                     c_ops=[qt.Qobj(Lk) for Lk in L_list])
    return np.array([np.asarray(s.full()) for s in res.states])